# TP Integrador — Modulación LoRa / FSCM

Grupo 8 — **Testigos de Vangelista**.

Integrantes:

- Castro, Victor José Arturo 
- Carreras, Leonor Victoria


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# TRANSMISOR — 1. Parámetros del PHY y del preámbulo

SF = 8                          # bits por símbolo
SamplingRate = 1_000_000        # tasa del SDR [muestras/s]
B = SamplingRate                # el chirp barre exactamente la tasa de muestreo
M = 2 ** SF                     # muestras por símbolo = símbolos posibles
T = 1 / B                       # período de chip [s]
Ts = M * T                      # duración de un símbolo [s]
k = np.arange(M)                # índice de chip

fc = 915e6                      # portadora dentro de la banda 900-920 MHz

n_up = 8                        # up-chirps del preámbulo
n_down = 2                      # down-chirps completos
cuarto_down = True              # último 1/4 de down-chirp
n_simbolos = 10                 # símbolos del payload

tramas_en_buffer = 2            # tramas por captura del receptor
NFLAT = 50000                   # tramas por punto del canal FLAT
NSELECTIVE = 50000              # tramas por punto del canal selectivo
snr_fer_db = np.arange(-12, 0, 1)   # SNR = Eb/N0 [dB], de -12 a -1
semilla = 2024

print(f"SF = {SF} | M = {M} | B = {B/1e6:.3f} MHz | fc = {fc/1e6:.0f} MHz")
print(f"Símbolo = {Ts*1e6:.1f} µs | Preámbulo = {(n_up + n_down)*M + (M//4 if cuarto_down else 0)} muestras")
print(f"FER: NFLAT = {NFLAT} | NSELECTIVE = {NSELECTIVE} tramas por punto")

In [ ]:
# TRANSMISOR — 2. De bits a símbolo (Ecuación 1)
def bits_a_simbolo(bits, SF=SF):
    """Convierte filas de bits (LSB primero) en símbolos enteros."""
    simbolos = []
    for fila in np.asarray(bits):
        fila = np.asarray(fila, dtype=np.int64).ravel()
        if fila.size != SF:
            raise ValueError("cada fila debe tener SF bits")
        simbolos.append(int(np.dot(fila, 2 ** np.arange(SF))))
    return np.asarray(simbolos)


generador_bits = np.random.default_rng(semilla)
bits_payload = generador_bits.integers(0, 2, size=(n_simbolos, SF))
simbolos_payload = bits_a_simbolo(bits_payload)

print("Bits del payload (LSB primero) -> símbolo:")
for fila, simbolo in zip(bits_payload, simbolos_payload):
    print(f"  {fila.tolist()} -> {simbolo:3d}")

In [ ]:
# TRANSMISOR — 3. Modulación de símbolo (Ecuaciones 2 y 3)
def up_chirp(M=M):
    """Chirp ascendente base, energía unitaria."""
    kk = np.arange(M)
    return (1.0 / np.sqrt(M)) * np.exp(1j * 2 * np.pi * (kk ** 2) / M)


def down_chirp(M=M):
    """Chirp descendente: el conjugado del ascendente."""
    return np.conj(up_chirp(M))


def down_chirp_demod(M=M):
    """Réplica usada para dechirpear (sin la escala 1/sqrt(M))."""
    kk = np.arange(M)
    return np.exp(-1j * 2 * np.pi * (kk ** 2) / M)


def modular(simbolos, M=M):
    """Genera la señal FSCM de una secuencia de símbolos."""
    kk = np.arange(M)
    partes = []
    for simbolo in np.asarray(simbolos).ravel():
        if not 0 <= int(simbolo) < M:
            raise ValueError(f"símbolo fuera de rango: {simbolo}")
        fase = 2 * np.pi * (((int(simbolo) + kk) % M) * kk) / M
        partes.append((1 / np.sqrt(M)) * np.exp(1j * fase))
    return np.concatenate(partes)


def preambulo(M=M, n_up=n_up, n_down=n_down, cuarto=cuarto_down):
    """Preámbulo: up-chirps, down-chirps y 1/4 de down-chirp."""
    partes = [np.tile(up_chirp(M), n_up)]
    if n_down:
        partes.append(np.tile(down_chirp(M), n_down))
    if cuarto:
        partes.append(down_chirp(M)[: M // 4])
    return np.concatenate(partes)


def longitud_preambulo(M=M, n_up=n_up, n_down=n_down, cuarto=cuarto_down):
    largo = n_up * M + n_down * M
    if cuarto:
        largo += M // 4
    return largo

In [ ]:
# TRANSMISOR — 4. Gráfico de la señal del preámbulo
def plotSignal(signal):
    #------------------------------- Componente en fase -------------------------------
    plt.figure(figsize=(20, 8), dpi=80, facecolor="w", edgecolor="k")
    plt.subplot(2, 1, 1)
    plt.plot(np.real(signal))
    plt.xlabel("n")
    plt.ylabel("Amplitud")
    plt.title("Componente en fase de la señal")
    plt.grid()
    #------------------------------- Componente en cuadratura ------------------------
    plt.subplot(2, 1, 2)
    plt.plot(np.imag(signal))
    plt.xlabel("n")
    plt.ylabel("Amplitud")
    plt.title("Componente en cuadratura de la señal")
    plt.grid()
    plt.show()


preambulo_ref = preambulo()
plotSignal(preambulo_ref)

**Preámbulo en el dominio del tiempo**

Se muestran las componentes en fase y en cuadratura del preámbulo. La amplitud se mantiene aproximadamente constante y lo que cambia es la frecuencia instantánea: el chirp barre la banda de forma lineal. La forma de onda corresponde a las ecuaciones (2)-(3) del paper, particularizadas para los símbolos del preámbulo. Sirve para verificar que la señal se construyó bien antes de agregar el payload.


In [ ]:
# TRANSMISOR — 5. Densidad espectral de potencia del preámbulo
def plotSpect(signal, samplingRate):
    plt.subplots(figsize=(15, 3))
    plt.grid()
    plt.title("Power Spectral Density")
    plt.psd(signal, NFFT=len(signal), Fs=samplingRate, color="C1")
    plt.show()


plotSpect(preambulo_ref, SamplingRate)

**Densidad espectral de potencia del preámbulo**

La PSD del preámbulo ocupa aproximadamente la banda de 1 MHz centrada en la portadora. El chirp recorre todas las frecuencias de la banda, por eso el espectro es bastante plano y no se concentra en unas pocas líneas. Es una consecuencia directa de la forma de onda de las ecuaciones (2)-(3). Confirma que la modulación no se sale de la banda útil.


In [ ]:
# TRANSMISOR — 6. Gráfico de la señal completa (trama)
trama = np.concatenate([preambulo_ref, modular(simbolos_payload)])
plotSignal(trama)

**Trama completa en el dominio del tiempo**

Ahora se grafica la trama completa: el preámbulo seguido por los símbolos del payload. Cada símbolo es un chirp con un desplazamiento cíclico de frecuencia, de acuerdo con la ecuación (1) para el símbolo y las ecuaciones (2)-(3) para la forma de onda. La envolvente sigue siendo aproximadamente constante; los cambios de fase entre símbolos son esperables. Esta es la señal que el transmisor repite de forma cíclica.


In [ ]:
# TRANSMISOR — 7. Densidad espectral de potencia de la trama completa
plotSpect(trama, SamplingRate)

**Densidad espectral de potencia de la trama**

La PSD de la trama completa es prácticamente igual a la del preámbulo: agregar payload no ensancha la banda ni genera componentes fuera de 1 MHz. El payload también está formado por chirps, así que la ocupación espectral queda determinada por las ecuaciones (2)-(3). Sirve para confirmar que la trama completa sigue siendo compatible con el ancho de banda del canal.


In [ ]:
# TRANSMISOR — 8. Demodulación (Ecuaciones 15 y 16)
def demodular(recibido, M=M):
    """Demodula uno o varios símbolos por dechirp + FFT."""
    datos = np.asarray(recibido, dtype=np.complex128)
    if datos.ndim == 1:
        if datos.size % M != 0:
            raise ValueError("la señal no es múltiplo de M")
        datos = datos.reshape(-1, M)
    dechirpeado = datos * down_chirp_demod(M)[None, :]
    espectros = np.abs(np.fft.fft(dechirpeado, axis=1))
    return np.argmax(espectros, axis=1)


simbolos_recuperados = demodular(modular(simbolos_payload))
print("Demodulación ida y vuelta:", np.array_equal(simbolos_recuperados, simbolos_payload))

In [ ]:
# TRANSMISOR — 9. Armado de la trama: preámbulo + payload
def construir_trama(simbolos, M=M, n_up=n_up, n_down=n_down, cuarto=cuarto_down):
    """Preámbulo seguido del payload modulado."""
    return np.concatenate([preambulo(M, n_up, n_down, cuarto), modular(simbolos, M)])


def longitud_trama(M=M, n_simbolos=n_simbolos, n_up=n_up, n_down=n_down,
                   cuarto=cuarto_down):
    return longitud_preambulo(M, n_up, n_down, cuarto) + n_simbolos * M


trama = construir_trama(simbolos_payload)
print(f"Preámbulo: {preambulo_ref.size} muestras")
print(f"Payload:   {n_simbolos*M} muestras")
print(f"Trama:     {trama.size} muestras ({trama.size/SamplingRate*1e3:.3f} ms)")

In [ ]:
# TRANSMISOR — 10. Detección del preámbulo: correlación en tiempo y en frecuencia
def correlacion_tiempo(recibido, patron=preambulo_ref):
    """Correlación directa en el dominio del tiempo."""
    recibido = np.asarray(recibido, dtype=np.complex128)
    patron = np.asarray(patron, dtype=np.complex128)
    if recibido.size < patron.size:
        raise ValueError("la señal es más corta que el patrón")
    return np.abs(np.correlate(recibido, patron, mode="valid"))


def correlacion_fft(recibido, patron=preambulo_ref, circular=False):
    """|correlación| calculada por FFT. Acepta una captura o un lote 2D."""
    recibido = np.asarray(recibido)
    patron = np.asarray(patron)
    if recibido.shape[-1] < patron.size:
        raise ValueError("la señal es más corta que el patrón")
    if circular:
        # El patrón puede ser más corto que la captura: se rellena con ceros a la
        # longitud de la captura para que la correlación circular tenga el mismo largo.
        producto = np.fft.fft(recibido, axis=-1) * np.conj(
            np.fft.fft(patron, recibido.shape[-1], axis=-1))
        return np.abs(np.fft.ifft(producto, axis=-1))
    n = recibido.shape[-1] + patron.size - 1
    nfft = 1 << (n - 1).bit_length()
    producto = np.fft.fft(recibido, nfft, axis=-1) * np.conj(np.fft.fft(patron, nfft))
    completo = np.fft.ifft(producto, axis=-1)
    return np.abs(completo[..., : recibido.shape[-1] - patron.size + 1])


def detectar_tiempo(recibido, patron=preambulo_ref):
    """Inicio y métrica del preámbulo por correlación temporal."""
    recibido = np.asarray(recibido, dtype=np.complex128)
    correlacion = correlacion_tiempo(recibido, patron)
    inicio = int(np.argmax(correlacion))
    energia = np.sum(np.abs(recibido[inicio: inicio + patron.size]) ** 2)
    metrica = correlacion[inicio] / (np.linalg.norm(patron) * np.sqrt(energia))
    return {"inicio": inicio, "metrica": float(metrica), "correlacion": correlacion}


def detectar_frecuencia(recibido, patron=preambulo_ref, circular=False):
    """Inicio y métrica del preámbulo por correlación en frecuencia (FFT)."""
    recibido = np.asarray(recibido)
    unico = recibido.ndim == 1
    lote = recibido.reshape(1, -1) if unico else recibido
    correlacion = correlacion_fft(lote, patron, circular)
    inicio = np.argmax(correlacion, axis=-1)
    if circular:
        if patron.size == lote.shape[-1]:
            energia = np.sum(np.abs(lote) ** 2, axis=-1)
        else:
            # Energía de la ventana del patrón en cada posición, con vuelta circular.
            potencia = np.abs(lote) ** 2
            doble = np.concatenate([potencia, potencia], axis=-1)
            acumulada = np.cumsum(
                np.concatenate([np.zeros((lote.shape[0], 1), dtype=potencia.dtype), doble],
                               axis=1), axis=1)
            filas = np.arange(lote.shape[0])
            energia = acumulada[filas, inicio + patron.size] - acumulada[filas, inicio]
    else:
        potencia = np.abs(lote) ** 2
        acumulada = np.cumsum(
            np.concatenate([np.zeros((lote.shape[0], 1), dtype=potencia.dtype), potencia], axis=1),
            axis=1)
        filas = np.arange(lote.shape[0])
        energia = acumulada[filas, inicio + patron.size] - acumulada[filas, inicio]
    metrica = correlacion[np.arange(lote.shape[0]), inicio] / (
        np.linalg.norm(patron) * np.sqrt(energia))
    if unico:
        return {"inicio": int(inicio[0]), "metrica": float(metrica[0]),
                "correlacion": correlacion[0]}
    return {"inicio": inicio, "metrica": metrica, "correlacion": correlacion}


# Demostración: la trama desplazada y con ruido debe dar el mismo inicio por los dos métodos.
margen = 64
captura_demo = np.zeros(trama.size + 2 * margen, dtype=np.complex128)
captura_demo[margen: margen + trama.size] = trama
generador_demo = np.random.default_rng(semilla + 1)
ruido_demo = (generador_demo.normal(size=captura_demo.size)
              + 1j * generador_demo.normal(size=captura_demo.size))
potencia_demo = np.mean(np.abs(trama) ** 2)
captura_demo += ruido_demo * np.sqrt(potencia_demo / 2)

info_tiempo = detectar_tiempo(captura_demo)
info_frecuencia = detectar_frecuencia(captura_demo)
print(f"Inicio real: {margen}")
print(f"Tiempo     : {info_tiempo['inicio']} (métrica {info_tiempo['metrica']:.3f})")
print(f"Frecuencia : {info_frecuencia['inicio']} (métrica {info_frecuencia['metrica']:.3f})")

In [ ]:
# TRANSMISOR — 11. FER en simulación contra SNR
# La captura es la trama completa (preámbulo + payload), pero solo se evalúa la
# detección del preámbulo. A confirmar: por ahora el payload no se demodula.
import os


def fer_simulada(selectivo, snr_db, n_tramas, lote=50, semilla=semilla):
    """Devuelve (fallas, hechas) para un bloque de n_tramas."""
    generador = np.random.default_rng(semilla)
    snr_muestra = snr_db - 10 * np.log10(M / SF)   # SNR referida a Eb/N0
    snr_lineal = 10 ** (snr_muestra / 10)
    patron = np.asarray(preambulo_ref, dtype=np.complex64)
    base = np.asarray(trama, dtype=np.complex64)
    # Respuesta al impulso del canal: unitaria para FLAT y de dos taps para selectivo.
    h = (np.array([np.sqrt(0.8), np.sqrt(0.2)], dtype=np.complex64)
         if selectivo else np.array([1.0], dtype=np.complex64))
    potencia_simbolo = 1 / M
    sigma_ruido = np.sqrt(potencia_simbolo / snr_lineal / 2)   # var gaussiana
    fallas = 0
    hechas = 0
    while hechas < n_tramas:
        b = min(lote, n_tramas - hechas)
        desplazamientos = generador.integers(0, base.size, size=b)
        capturas = np.empty((b, base.size), dtype=np.complex64)
        for i, desplazamiento in enumerate(desplazamientos):
            captura = np.roll(base, desplazamiento)
            # Canal simulado por convolución con h, frame por frame.
            capturas[i] = np.convolve(captura, h)[:base.size]
        ruido = (generador.normal(size=capturas.shape)
                 + 1j * generador.normal(size=capturas.shape)).astype(np.complex64)
        capturas += sigma_ruido * ruido
        inicio = detectar_frecuencia(capturas, patron, circular=True)["inicio"]
        fallas += int(np.count_nonzero(inicio != desplazamientos))
        hechas += b
    return fallas, hechas


# NFLAT y NSELECTIVE son fijas: 50 000 tramas por punto en cada canal.
# El checkpoint guarda fallas/hechas cada 500 tramas para poder retomar.
lote_fer = 50
ruta_checkpoint = "fer_checkpoint.npz"
version_checkpoint = 2          # subir si cambia el canal, el ruido o la captura

n_puntos = snr_fer_db.size
datos = None
if os.path.exists(ruta_checkpoint):
    datos = np.load(ruta_checkpoint)
    if ("version" not in datos.files
            or int(datos["version"]) != version_checkpoint
            or datos["fallas_flat"].size != n_puntos):
        datos = None

if datos is None:
    fallas_flat = np.zeros(n_puntos, dtype=int)
    hechas_flat = np.zeros(n_puntos, dtype=int)
    fallas_selectivo = np.zeros(n_puntos, dtype=int)
    hechas_selectivo = np.zeros(n_puntos, dtype=int)
else:
    fallas_flat = datos["fallas_flat"].astype(int)
    hechas_flat = datos["hechas_flat"].astype(int)
    fallas_selectivo = datos["fallas_selectivo"].astype(int)
    hechas_selectivo = datos["hechas_selectivo"].astype(int)

canales = [
    ("FLAT", False, NFLAT, fallas_flat, hechas_flat),
    ("selectivo", True, NSELECTIVE, fallas_selectivo, hechas_selectivo),
]

for nombre, selectivo, objetivo, fallas, hechas in canales:
    for i, snr in enumerate(snr_fer_db):
        while hechas[i] < objetivo:
            n_bloque = min(500, objetivo - hechas[i])
            f, n = fer_simulada(selectivo, snr, n_bloque, lote=lote_fer,
                                semilla=semilla + 100000 * i + hechas[i])
            fallas[i] += f
            hechas[i] += n
            ruta_temporal = ruta_checkpoint + ".tmp.npz"
            np.savez(ruta_temporal,
                     version=version_checkpoint,
                     fallas_flat=fallas_flat, hechas_flat=hechas_flat,
                     fallas_selectivo=fallas_selectivo, hechas_selectivo=hechas_selectivo)
            os.replace(ruta_temporal, ruta_checkpoint)
            if hechas[i] % 10000 == 0 or hechas[i] >= objetivo:
                print(f"{nombre:9s} SNR {snr:>3.0f} dB: {fallas[i]:5d}/{hechas[i]:5d} "
                      f"= {fallas[i] / max(hechas[i], 1):.5f}")
        print(f"{nombre:9s} SNR {snr:>3.0f} dB listo: {fallas[i]}/{hechas[i]} "
              f"= {fallas[i] / max(hechas[i], 1):.5f}")

fer_flat = fallas_flat / np.maximum(hechas_flat, 1)
fer_selectivo = fallas_selectivo / np.maximum(hechas_selectivo, 1)

print(f"{'SNR':>5} {'FER FLAT':>10} {'N FLAT':>8} {'FER SEL':>10} {'N SEL':>8}")
for snr, ff, hf, fs, hs in zip(snr_fer_db, fer_flat, hechas_flat,
                               fer_selectivo, hechas_selectivo):
    print(f"{snr:>5.0f} {ff:>10.5f} {hf:>8d} {fs:>10.5f} {hs:>8d}")

piso = 1 / (2 * max(NFLAT, NSELECTIVE))
plt.figure(figsize=(7, 4.5))
plt.semilogy(snr_fer_db, np.maximum(fer_flat, piso), "o-", label="FLAT (solo ruido)")
plt.semilogy(snr_fer_db, np.maximum(fer_selectivo, piso), "s-", label="selectivo (2 taps)")
plt.xlabel("SNR = Eb/N0 [dB]")
plt.ylabel("FER")
plt.title("FER de la trama (detección del preámbulo)")
plt.xticks(snr_fer_db)
plt.grid(True, which="both", alpha=0.3)
plt.legend()
plt.show()

**FER de la trama contra `Eb/N0`**

Se compara la FER de la trama en dos canales: FLAT (solo ruido) y selectivo en frecuencia. La simulación es offline, no usa el SDR: las tramas se generan a mano. El canal se modela **por convolución** con la respuesta al impulso, frame por frame: `h = [1]` para FLAT y `h = [√0.8, √0.2]` para selectivo (dos taps, eco a una muestra, energía unitaria). El ruido es gaussiano complejo de media cero, con `Pz = (1/M)/SNR` (ecuación (10)). La detección usa solo el preámbulo como patrón; el payload está presente pero no se demodula. El receptor óptimo del paper se apoya en las ecuaciones (11)-(16). Se usan **NFLAT = NSELECTIVE = 50 000 tramas por punto** y un checkpoint cada 500 tramas. El gráfico muestra los 12 valores de SNR, de −12 a −1. La curva se actualiza al correr; con 50 000 tramas por punto la resolución es 1/50 000. No es una reproducción de la Figura 1: el paper mide BER con SF = 7 y acá se mide FER de detección con SF = 8.

In [ ]:
# CONFIGURACIÓN DEL SDR — se deja el enlace abierto para el receptor
import adi

sdr = adi.Pluto("ip:192.168.1.34")

# Si otra sesión dejó el chip en "alert", el RX no entrega muestras.
if sdr._ctrl.attrs["ensm_mode"].value != "fdd":
    sdr._ctrl.attrs["ensm_mode"].value = "fdd"

sdr.loopback = 0
sdr.sample_rate = int(SamplingRate)
sdr.tx_rf_bandwidth = 2_000_000
sdr.rx_rf_bandwidth = 2_000_000
sdr.tx_hardwaregain_chan0 = -60
sdr.gain_control_mode_chan0 = "slow_attack"
sdr.tx_lo = int(fc)
sdr.rx_lo = int(fc)
sdr.rx_buffer_size = tramas_en_buffer * longitud_trama()
sdr.tx_cyclic_buffer = True

print(f"Enlace configurado: {fc/1e6:.0f} MHz | RX buffer {sdr.rx_buffer_size} muestras")
print("El buffer cíclico se activa antes de transmitir.")

In [ ]:
# ENVÍO DE LA SEÑAL AL SDR
sdr.tx(trama * (2 ** 15 - 1))
print("Trama transmitida; el buffer cíclico la repite sin pausa.")

In [ ]:
# RECEPTOR — 1. Gráfico de la señal recibida
captura = sdr.rx()
plotSignal(captura)

**Señal recibida en el dominio del tiempo**

Se grafica la captura real del receptor: dos tramas consecutivas con preámbulo, payload y ruido. La amplitud depende de la ganancia y del AGC del Pluto. La señal recibida responde al modelo de la ecuación (10). Es la entrada que después usa `detectar_frecuencia` para buscar el preámbulo; a simple vista el preámbulo puede quedar tapado por el ruido, y por eso la detección se hace por correlación.


In [ ]:
# RECEPTOR — 2. Densidad espectral de potencia de la señal recibida
plotSpect(captura, SamplingRate)

**Densidad espectral de potencia de la señal recibida**

La PSD de la captura muestra la señal ocupando aproximadamente 1 MHz y un piso de ruido repartido en la banda. Es la versión experimental de la ecuación (10): señal más ruido blanco. Comparada con la PSD de la celda 8, permite ver cuánto ruido agregó el enlace y si la señal sigue dentro del ancho de banda esperado.


In [ ]:
# RECEPTOR — 3. FER con 20000 tramas (sin gráfico)
n_tramas_enlace = 20000
umbral_deteccion = 0.5
tramas_ok = 0
tramas_mal = 0

for indice in range(n_tramas_enlace):
    captura = sdr.rx()                 # 2 tramas: preámbulo + payload
    # El patrón es solo el preámbulo; la búsqueda recorre toda la captura.
    info = detectar_frecuencia(captura)
    if info["metrica"] >= umbral_deteccion:
        tramas_ok += 1
    else:
        tramas_mal += 1
    if (indice + 1) % 2000 == 0:
        print(f"{indice + 1}/{n_tramas_enlace} tramas | errores: {tramas_mal}")

fer_enlace = tramas_mal / n_tramas_enlace
print(f"FER con {n_tramas_enlace} tramas: {fer_enlace:.6f}")
print(f"  detectadas: {tramas_ok} | falladas: {tramas_mal}")

In [ ]:
# CIERRE DEL ENLACE — dejar el Pluto atenuado y fuera de la banda
sdr.tx_destroy_buffer()
sdr.tx_hardwaregain_chan0 = -89
sdr.tx_lo = 2400e6
sdr.rx_lo = 950e6
sdr.tx(np.zeros(2048, dtype=np.complex128))
print("Enlace cerrado: TX atenuado y portadoras fuera de la banda.")